# BERT-QPP$_{cross}$ training on MS MARCO dev — via the repo's own scripts

Fine-tunes a `bert-base-uncased` cross-encoder (same architecture as `train_CE.py` always used) to predict per-query
retrieval performance, using **all** MS MARCO **dev** queries as the training set instead of train (no held-out
validation split). Wired to call the repo's own scripts at every step, the same convention as
`BERTQPP_TREC_DL_colab_repo_scripts.ipynb`:

- **`create_map.py`** — per-query MRR@10 (via `pytrec_eval`) from a BM25 run + qrels. Target label for training.
- **`create_train_query.py`** — joins that per-query metric with query text.
- **`create_train_pkl_file.py`** — attaches each query's top-retrieved doc text from a passage collection, pickles
  `{qid: {qtext, doc_text, performance}}`.
- **`train_CE.py`** — fine-tunes the cross-encoder on that pickle.

Sanity-check cells are interleaved throughout to catch bad downloads, a mismatched BM25 run, or missing doc text
before they silently corrupt the training set.

**Fully self-contained — nothing to supply manually.** The notebook downloads:
- `queries.dev.small.tsv` / `qrels.dev.small.tsv` (MD5-verified, same multi-mirror logic as the eval notebooks)
- `top1000.dev.tsv` — the **official BM25 top-1000 run** MS MARCO releases for exactly these 6,980 dev-small
  queries (`qid`, `pid`, `query`, `passage` per row). No published checksum exists for this file, so it's validated
  functionally (row/query counts, column shape) instead of by MD5.

From `top1000.dev.tsv` the notebook derives two things itself, so there's no need to download the full 8.8M-passage
`collection.tsv`:
- a standard 6-column TREC run (`DEV_RUN_PATH`) — rank taken from each query's position in the file, score a
  synthetic descending value that preserves that BM25 ordering
- a passage collection (`COLLECTION_PATH`) containing just the passages that actually appear in that run, keyed
  by passage id — a free byproduct of parsing the run, and all `create_train_pkl_file.py` needs to look up doc text


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone https://github.com/Riddhi2587/BERTQPP.git /content/BERTQPP
%cd /content/BERTQPP

!pip install -q sentence-transformers pytrec_eval pandas tqdm

In [ ]:
import os

# ---- Paths on Drive ----
DRIVE_ROOT = "/content/drive/MyDrive/precise-qpp"
DATA_DIR = f"{DRIVE_ROOT}/data"
OUT_DIR = f"{DRIVE_ROOT}/results/bertqpp_train_msmarco_dev"
MODEL_OUT_DIR = f"{DRIVE_ROOT}/models/bertqpp_train_msmarco_dev"

DEV_QUERIES_PATH = f"{DATA_DIR}/queries.dev.small.tsv"
DEV_QRELS_PATH = f"{DATA_DIR}/qrels.dev.small.tsv"

# Raw download cache for the official BM25 top-1000 run over the dev-small queries.
# DEV_RUN_PATH and COLLECTION_PATH are derived from this file further down.
TOP1000_RUN_PATH = f"{DATA_DIR}/top1000.dev.tsv"

# ---- Training config ----
CHECKPOINT = "bert-base-uncased"   # starting model for the cross-encoder
METRIC = "MRR"                     # MRR or MAP (see create_map.py)
CUTOFF = 10                        # -> MRR@10
BATCH_SIZE = 16
EPOCHS = 1

for d in [DATA_DIR, OUT_DIR, MODEL_OUT_DIR]:
    os.makedirs(d, exist_ok=True)

## Download the MS MARCO dev queries and dev qrels

Same MD5-verified, multi-mirror download as the eval notebooks (the primary `msmarco.blob.core.windows.net` host
returns 409; `z22.web.core.windows.net` is the same dataset's still-public static-website endpoint, Dropbox is a
community mirror). Skips the download if both files are already cached on Drive.

`collection.tsv` is deliberately not downloaded here — the BM25 run pulled in the next section already carries
the passage text for every doc it references, which is all `create_train_pkl_file.py` needs.

In [ ]:
import subprocess

MSMARCO_URLS = [
    "https://msmarco.z22.web.core.windows.net/msmarcoranking/collectionandqueries.tar.gz",
    "https://www.dropbox.com/s/9f54jg2f71ray3b/collectionandqueries.tar.gz?dl=1",
]
MSMARCO_MD5 = "31644046b18952c1386cd4564ba2ae69"
ARCHIVE_MEMBERS = {
    DEV_QUERIES_PATH: "queries.dev.small.tsv",
    DEV_QRELS_PATH: "qrels.dev.small.tsv",
}

missing = {out: member for out, member in ARCHIVE_MEMBERS.items() if not os.path.exists(out)}

if missing:
    archive = "/content/collectionandqueries.tar.gz"

    verified = False
    for url in MSMARCO_URLS:
        print(f"[INFO] Trying {url}")
        subprocess.run(["rm", "-f", archive])
        dl = subprocess.run(["bash", "-c", f'wget -q --show-progress -O "{archive}" "{url}"'])
        if dl.returncode != 0:
            print(f"[WARN] Download failed from {url}, trying next source")
            continue
        md5 = subprocess.run(["md5sum", archive], capture_output=True, text=True).stdout.split()[0]
        if md5 != MSMARCO_MD5:
            print(f"[WARN] MD5 mismatch from {url} (got {md5}, expected {MSMARCO_MD5}), trying next source")
            continue
        verified = True
        break

    if not verified:
        raise RuntimeError(
            f"Could not download a valid collectionandqueries.tar.gz from any known source ({MSMARCO_URLS}). "
            f"Download it manually and place queries.dev.small.tsv / qrels.dev.small.tsv under {DATA_DIR}."
        )

    for out_path, member in missing.items():
        os.makedirs(os.path.dirname(out_path), exist_ok=True)
        extract = subprocess.run(["bash", "-c",
            f'tar -xzf "{archive}" -C /content {member} && mv /content/{member} "{out_path}"'], capture_output=True, text=True)
        if extract.returncode != 0:
            raise RuntimeError(f"Archive was verified but extracting {member} from it failed:\n{extract.stderr}")
    subprocess.run(["rm", "-f", archive])
else:
    print("[INFO] Using cached queries/qrels on Drive")

print(f"dev queries: {DEV_QUERIES_PATH}")
print(f"dev qrels:   {DEV_QRELS_PATH}")

## Sanity check: dataset sizes and query/qrels coverage

In [ ]:
def count_lines(path):
    with open(path) as f:
        return sum(1 for _ in f)

n_queries = count_lines(DEV_QUERIES_PATH)
n_qrels = count_lines(DEV_QRELS_PATH)

print(f"[CHECK] queries.dev.small.tsv: {n_queries:,} queries")
print(f"[CHECK] qrels.dev.small.tsv:   {n_qrels:,} qrel rows")

assert n_queries > 0, "queries.dev.small.tsv is empty"
assert n_qrels > 0, "qrels.dev.small.tsv is empty"

query_ids = set()
with open(DEV_QUERIES_PATH) as f:
    for line in f:
        qid, _ = line.rstrip("\n").split("\t", 1)
        query_ids.add(qid)

qrel_qids = set()
with open(DEV_QRELS_PATH) as f:
    for line in f:
        parts = line.split()
        if parts:
            qrel_qids.add(parts[0])

judged = query_ids & qrel_qids
print(f"[CHECK] {len(judged):,} / {len(query_ids):,} dev queries have a qrel ({len(judged) / len(query_ids):.1%})")

assert judged, "No overlap between dev query ids and dev qrel ids -- wrong files, or a qid format mismatch?"

## Download the official BM25 top-1000 run for the dev queries

MS MARCO releases its own BM25 top-1000 retrieval results for exactly the dev-small queries as `top1000.dev.tar.gz`
(`qid`, `pid`, `query`, `passage` per row, ~6.67M rows). No published checksum exists for this specific archive
(unlike `collectionandqueries.tar.gz`), so it's downloaded with a size check as a crude corruption guard and
verified functionally in the next cell instead of by MD5. Tries the working `z22.web.core.windows.net` mirror
first, falls back to the legacy `blob.core.windows.net` host in case only that specific object is still public.

In [ ]:
TOP1000_URLS = [
    "https://msmarco.z22.web.core.windows.net/msmarcoranking/top1000.dev.tar.gz",
    "https://msmarco.blob.core.windows.net/msmarcoranking/top1000.dev.tar.gz",
]

def extract_top1000(archive_path, output_path):
    """Extract the run file from top1000.dev.tar.gz, whatever it's actually named/nested as inside --
    don't assume the member name matches the archive name."""
    listing = subprocess.run(["tar", "-tzf", archive_path], capture_output=True, text=True)
    if listing.returncode != 0:
        raise RuntimeError(f"Could not list contents of {archive_path}:\n{listing.stderr}")

    members = [m for m in listing.stdout.splitlines() if m and not m.endswith("/")]
    print(f"[INFO] Archive contains: {members}")

    candidates = [m for m in members if "top1000.dev" in m] or members
    if len(candidates) != 1:
        raise RuntimeError(
            f"Expected exactly one relevant member in {archive_path}, found {candidates}. "
            "Inspect the archive and adjust extract_top1000() accordingly."
        )
    member = candidates[0]

    extract = subprocess.run(["tar", "-xzf", archive_path, "-C", "/content", member], capture_output=True, text=True)
    if extract.returncode != 0:
        raise RuntimeError(f"Extracting {member} from {archive_path} failed:\n{extract.stderr}")

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    move = subprocess.run(["mv", f"/content/{member}", output_path], capture_output=True, text=True)
    if move.returncode != 0:
        raise RuntimeError(f"Moving /content/{member} to {output_path} failed:\n{move.stderr}")


if not os.path.exists(TOP1000_RUN_PATH):
    archive = "/content/top1000.dev.tar.gz"
    downloaded = False

    for url in TOP1000_URLS:
        print(f"[INFO] Trying {url}")
        subprocess.run(["rm", "-f", archive])
        dl = subprocess.run(["bash", "-c", f'wget -q --show-progress -O "{archive}" "{url}"'])
        if dl.returncode == 0 and os.path.exists(archive) and os.path.getsize(archive) > 10_000_000:
            downloaded = True
            break
        print(f"[WARN] Download from {url} failed or looked too small, trying next source")

    if not downloaded:
        raise RuntimeError(
            "Could not download top1000.dev.tar.gz from any known host. No published checksum exists for this "
            "file, so if you already have it, download it manually and place top1000.dev.tsv at "
            f"{TOP1000_RUN_PATH}."
        )

    extract_top1000(archive, TOP1000_RUN_PATH)
    subprocess.run(["rm", "-f", archive])
else:
    print(f"[INFO] Using cached {TOP1000_RUN_PATH}")

## Sanity check: raw top1000.dev.tsv

In [ ]:
n_top1000_rows = count_lines(TOP1000_RUN_PATH)
print(f"[CHECK] top1000.dev.tsv: {n_top1000_rows:,} rows")

assert n_top1000_rows > 1_000_000, f"top1000.dev.tsv looks too small ({n_top1000_rows:,} rows) -- possibly a truncated/corrupt download"

with open(TOP1000_RUN_PATH) as f:
    first_line = f.readline()
first_cols = first_line.rstrip("\n").split("\t")
print(f"[CHECK] first row has {len(first_cols)} columns (expect 4: qid, pid, query, passage)")
assert len(first_cols) == 4, f"Expected 4 tab-separated columns, got: {first_line[:200]!r}"

## Derive the training BM25 run and passage collection from top1000.dev.tsv

`top1000.dev.tsv` has no explicit rank or score column -- a query's rows are in BM25 rank order by construction
(the file is generated per query), so rank is each row's position within its query's block, and score is a
synthetic descending value that preserves that ordering when `create_map.py` re-ranks by score. The passage text
already in each row is written out once per unique `pid` as a small collection covering only the docs this run
actually references.

In [ ]:
from collections import defaultdict

DEV_RUN_PATH = f"{OUT_DIR}/dev_bm25_top1000.run"
COLLECTION_PATH = f"{OUT_DIR}/dev_collection_subset.tsv"

rank_counters = defaultdict(int)
seen_pids = set()
run_rows_written = 0

with open(TOP1000_RUN_PATH) as fin, \
     open(DEV_RUN_PATH, "w") as run_out, \
     open(COLLECTION_PATH, "w") as coll_out:
    for line in fin:
        qid, pid, _query, passage = line.rstrip("\n").split("\t")

        rank_counters[qid] += 1
        rank = rank_counters[qid]
        score = 1_000_000 - rank
        run_out.write(f"{qid}\tQ0\t{pid}\t{rank}\t{score}\tbm25-top1000-dev\n")
        run_rows_written += 1

        if pid not in seen_pids:
            seen_pids.add(pid)
            coll_out.write(f"{pid}\t{passage}\n")

print(f"[INFO] DEV_RUN_PATH:    {run_rows_written:,} rows across {len(rank_counters):,} queries")
print(f"[INFO] COLLECTION_PATH: {len(seen_pids):,} unique passages")

assert len(rank_counters) > 0, "No queries found in top1000.dev.tsv"
assert len(seen_pids) > 1000, f"Derived collection looks too small ({len(seen_pids)} passages)"

## Sanity check: BM25 run file

In [ ]:
run_rows = 0
bad_rows = 0
run_qids = set()
ranks_seen = set()

with open(DEV_RUN_PATH) as f:
    for line in f:
        run_rows += 1
        parts = line.split()
        if len(parts) < 6:
            bad_rows += 1
            continue
        qid, _, docid, rank, score, _ = parts[:6]
        run_qids.add(qid)
        ranks_seen.add(int(rank))
        float(score)  # raises if the score column isn't numeric

print(f"[CHECK] BM25 run: {run_rows:,} rows, {len(run_qids):,} unique queries, {bad_rows} malformed rows")
print(f"[CHECK] rank column ranges {min(ranks_seen)}-{max(ranks_seen)} (should start at 1)")

run_overlap = run_qids & query_ids
print(f"[CHECK] {len(run_overlap):,} / {len(query_ids):,} dev queries appear in the BM25 run ({len(run_overlap) / len(query_ids):.1%})")

assert bad_rows == 0, f"{bad_rows} rows in the run file don't have 6 whitespace-separated columns -- check the format"
assert run_overlap, "None of the dev queries appear in the BM25 run -- wrong run file, or a qid format mismatch?"
assert min(ranks_seen) == 1, f"Lowest rank seen is {min(ranks_seen)}, expected ranks to start at 1"

## Derive each query's top-1 doc from the BM25 run

`create_train_pkl_file.py` expects a reduced `qid<TAB>docid<TAB>rank` file (one row per query), not a full
top-*k* TREC run. Top-1 is picked by the lowest `rank` column, not by file order, so this is robust to
unsorted runs -- same logic as the eval notebooks' `parse_run`.

In [ ]:
def parse_run_top1(path):
    best_rank = {}
    top1 = {}
    with open(path) as f:
        for line in f:
            parts = line.split()
            if len(parts) < 6:
                continue
            qid, _, docid, rank, _, _ = parts[:6]
            rank = int(rank)
            if qid not in best_rank or rank < best_rank[qid]:
                best_rank[qid] = rank
                top1[qid] = docid
    return top1

top1 = parse_run_top1(DEV_RUN_PATH)
assert top1, f"No usable rows parsed from {DEV_RUN_PATH} -- check it's a standard 6-column TREC run file."
print(f"[INFO] Parsed top-1 doc for {len(top1)} queries from {DEV_RUN_PATH}")

TOP1_RUN_PATH = f"{OUT_DIR}/dev_top1.tsv"
with open(TOP1_RUN_PATH, "w") as f:
    for qid, docid in top1.items():
        f.write(f"{qid}\t{docid}\t1\n")

## Build the training label: per-query MRR@10 from the BM25 run + dev qrels

In [ ]:
METRIC_COL = f"{METRIC}@{CUTOFF}"
PERQUERY_METRIC_CSV = f"{OUT_DIR}/perquery_{METRIC.lower()}{CUTOFF}.csv"

!python3 create_map.py \
    --run "{DEV_RUN_PATH}" \
    --qrels "{DEV_QRELS_PATH}" \
    --metric {METRIC} \
    --cutoff {CUTOFF} \
    --output "{PERQUERY_METRIC_CSV}"

In [ ]:
QUERY_MAP_TSV = f"{OUT_DIR}/dev_query_{METRIC.lower()}{CUTOFF}.tsv"

!python3 create_train_query.py \
    --map-file "{PERQUERY_METRIC_CSV}" \
    --query-file "{DEV_QUERIES_PATH}" \
    --metric-col "{METRIC_COL}" \
    --output "{QUERY_MAP_TSV}"

## Sanity check: training metric score range

All dev queries are used for training (no held-out split) -- this just verifies the labels `train_CE.py`
will train against are in a sane range before spending time on fine-tuning.

In [ ]:
import pandas as pd

query_map_df = pd.read_csv(QUERY_MAP_TSV, sep="\t", names=["qid", "query", METRIC_COL], dtype={"qid": str})

print(f"[CHECK] {METRIC_COL} over {len(query_map_df):,} training queries:")
print(query_map_df[METRIC_COL].describe())

assert query_map_df[METRIC_COL].between(0, 1).all(), f"{METRIC_COL} values outside [0, 1] -- check the metric computation"

In [ ]:
TRAIN_PKL = f"{OUT_DIR}/train.pkl"

!python3 create_train_pkl_file.py \
    --collection "{COLLECTION_PATH}" \
    --query-map "{QUERY_MAP_TSV}" \
    --run "{TOP1_RUN_PATH}" \
    --output "{TRAIN_PKL}"

## Sanity check: top-1 doc text coverage

In [ ]:
import pickle

with open(TRAIN_PKL, "rb") as f:
    train_examples = pickle.load(f)

n_total = len(train_examples)
n_empty_text = sum(1 for v in train_examples.values() if not v["doc_text"].strip())
print(f"[CHECK] {n_total:,} training examples, {n_empty_text:,} with empty doc_text ({n_empty_text / n_total:.1%})")

sample_qid = next(iter(train_examples))
sample = train_examples[sample_qid]
print(f"[CHECK] sample qid={sample_qid}")
print(f"        query: {sample['qtext']}")
print(f"        doc:   {sample['doc_text'][:200]}")
print(f"        label: {sample['performance']}")

assert n_empty_text / n_total < 0.05, f"{n_empty_text}/{n_total} examples have empty doc_text -- check docid/collection alignment"

## Fine-tune the cross-encoder

In [ ]:
!python3 train_CE.py \
    --train-pkl "{TRAIN_PKL}" \
    --checkpoint "{CHECKPOINT}" \
    --batch-size {BATCH_SIZE} \
    --epochs {EPOCHS} \
    --output "{MODEL_OUT_DIR}"

## Next steps

The trained model is saved to `MODEL_OUT_DIR` on Drive. To evaluate it on TREC DL 19/20, point
`MODEL_PATH` in `BERTQPP_TREC_DL_colab_repo_scripts.ipynb` at this same path and run that notebook.